# Parameter Golf — Step 3.1: Architecture Experiments

Two parallel tracks exploring fundamentally different architectures:

**Track A: Improved Depth-Recurrent Transformer** — Fix PR #187 failures with LoRA adapters, NoPE, input injection. Patches train_gpt.py.

**Track B: Mamba-3 Prototype** — New SSM-based architecture. Zero competition attempts in 200+ PRs. Needs `mamba-ssm` package.

Same 2000 iters / 5 shards as Steps 2/3 for fair comparison.

## 1. Install Dependencies

In [ ]:
!pip install -q torch numpy tqdm huggingface-hub sentencepiece
!pip install -q mamba-ssm causal-conv1d

## 2. Clone Repo & Download Data

In [ ]:
import os

REPO_DIR = "/content/parameter-golf"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/openai/parameter-golf.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Download training shards + validation + tokenizer
# 5 shards (~1GB) for fast directional experiments. Increase for final runs (max 80).
TRAIN_SHARDS = 5

!python data/cached_challenge_fineweb.py --train-shards {TRAIN_SHARDS}

## 3. Detect GPU & Configure Hyperparameters

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > GPU")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
compute_cap = torch.cuda.get_device_capability(0)
supports_flash = compute_cap[0] >= 8  # Ampere+ (sm80)

print(f"GPU: {gpu_name}")
print(f"Memory: {gpu_mem_gb:.1f} GB")
print(f"Compute capability: {compute_cap[0]}.{compute_cap[1]}")
print(f"Flash attention: {'yes' if supports_flash else 'no (will use mem_efficient)'}")
print()

In [ ]:
# ============================================================
# STEP 2 CONFIG: Build on Step 1 best result (combined_best)
# ============================================================

# Load Step 1 results
import json as jsonlib
import glob as globmod

STEP1_DIR = "experiments"
DRIVE_DIR = "/content/drive/MyDrive/parameter-golf-experiments"

step1_results = {}
for base_dir in [STEP1_DIR, DRIVE_DIR]:
    if not os.path.exists(base_dir):
        continue
    for fname in sorted(globmod.glob(f"{base_dir}/*/result.json")):
        with open(fname) as f:
            r = jsonlib.load(f)
        step1_results[r["experiment"]] = r

if step1_results:
    ranked = sorted(step1_results.values(), key=lambda r: r.get("val_bpb", 999))
    print("Step 1 Results:")
    print(f"{'Experiment':<22} {'BPB':>8}")
    print("-" * 32)
    for r in ranked:
        print(f"{r['experiment']:<22} {r.get('val_bpb', 0):>8.4f}")
    print(f"\nBest: {ranked[0]['experiment']} (BPB={ranked[0].get('val_bpb', '?')})")
else:
    print("No Step 1 results found. Using default combined_best config.")

# Base config = Step 1 winner (combined_best)
BASE_CONFIG = {
    "NUM_LAYERS":         "10",
    "MLP_MULT":           "3",
    "MODEL_DIM":          "512",
    "NUM_HEADS":          "8",
    "NUM_KV_HEADS":       "4",
    "TRAIN_SEQ_LEN":      "2048",
    "MATRIX_LR":          "0.02",
    "SCALAR_LR":          "0.02",
    "TIED_EMBED_LR":      "0.03",
    "WARMDOWN_ITERS":     "800",
    "MUON_MOMENTUM":      "0.99",
    "MUON_MOMENTUM_WARMUP_START": "0.92",
    "MUON_MOMENTUM_WARMUP_STEPS": "500",
    "GRAD_CLIP_NORM":     "0.3",
}

# GPU-specific batch settings
if gpu_mem_gb >= 70:    PROFILE = "h100"
elif gpu_mem_gb >= 35:  PROFILE = "a100"
elif gpu_mem_gb >= 20:  PROFILE = "l4"
else:                   PROFILE = "t4"

BATCH_SETTINGS = {
    "t4":   {"TRAIN_BATCH_TOKENS": "131072",  "VAL_BATCH_SIZE": "131072"},
    "l4":   {"TRAIN_BATCH_TOKENS": "524288",  "VAL_BATCH_SIZE": "262144"},
    "a100": {"TRAIN_BATCH_TOKENS": "262144",  "VAL_BATCH_SIZE": "262144"},  # halved for speed
    "h100": {"TRAIN_BATCH_TOKENS": "524288",  "VAL_BATCH_SIZE": "524288"},
}

FAST_SETTINGS = {
    "ITERATIONS":           "2000",
    "WARMDOWN_ITERS":       "400",
    "MAX_WALLCLOCK_SECONDS": "600",
    "VAL_LOSS_EVERY":       "500",
    "TRAIN_LOG_EVERY":      "100",
}

print(f"\nStep 2 base: combined_best + {PROFILE} batch settings")
print(f"Fast mode: {FAST_SETTINGS['ITERATIONS']} iterations")

## 4. Patch train_gpt.py for Single-GPU Speed

In [ ]:
# Patch train_gpt.py for single-GPU speed:
# 1. Flash SDP fallback for T4/older GPUs
# 2. Reduce grad_accum from 8 to 4 → 2x faster steps, better VRAM usage

def apply_base_patches():
    with open("train_gpt.py", "r") as f:
        code = f.read()
    patched = False

    # Patch 1: SDP backend fallback (T4 only)
    if not supports_flash:
        old_sdp = """    enable_cudnn_sdp(False)
    enable_flash_sdp(True)
    enable_mem_efficient_sdp(False)
    enable_math_sdp(False)"""
        new_sdp = """    enable_cudnn_sdp(False)
    enable_flash_sdp(False)
    enable_mem_efficient_sdp(True)
    enable_math_sdp(True)"""
        if old_sdp in code:
            code = code.replace(old_sdp, new_sdp)
            print("Patched: flash_sdp -> mem_efficient_sdp (non-flash GPU)")
            patched = True

    # Patch 2: Reduce grad_accum_steps for single GPU
    GRAD_ACCUM = 8  # keep original — torch.compile disabled makes steps fast enough

    old_check = '    if 8 % world_size != 0:\n        raise ValueError(f"WORLD_SIZE={world_size} must divide 8 so grad_accum_steps stays integral")\n    grad_accum_steps = 8 // world_size'
    new_check = f'    grad_accum_steps = {GRAD_ACCUM}  # patched: was 8//world_size'
    if old_check in code:
        code = code.replace(old_check, new_check)
        print(f"Patched: grad_accum_steps = {GRAD_ACCUM} (was 8, 2x faster)")
        patched = True

    old_scale = "    grad_scale = 1.0 / grad_accum_steps"
    new_scale = f"    grad_scale = 1.0 / {GRAD_ACCUM}  # patched"
    if old_scale in code:
        code = code.replace(old_scale, new_scale)

    # Patch 3: Disable torch.compile (saves 5-10 min compilation per experiment)
    old_compile = "    compiled_model = torch.compile(base_model, dynamic=False, fullgraph=True)"
    new_compile = "    compiled_model = base_model  # torch.compile disabled for fast experiments"
    if old_compile in code:
        code = code.replace(old_compile, new_compile)
        print("Patched: torch.compile disabled (faster startup)")
        patched = True

    # Also disable Newton-Schulz compilation
    old_ns = "    zeropower_via_newtonschulz5 = torch.compile(zeropower_via_newtonschulz5)"
    new_ns = "    # zeropower_via_newtonschulz5 = torch.compile(zeropower_via_newtonschulz5)  # disabled"
    if old_ns in code:
        code = code.replace(old_ns, new_ns)

    if patched:
        with open("train_gpt.py", "w") as f:
            f.write(code)
    else:
        print("No patches needed (already applied or script changed)")

apply_base_patches()

## 5. Track A: Improved Depth-Recurrent Transformer

Fix PR #187's failures: add per-loop LoRA adapters, NoPE in shared blocks, input re-injection.

### Track A: Patch Functions

In [ ]:
import subprocess, math

def reset_script():
    subprocess.run(["git", "checkout", "train_gpt.py"], check=True, capture_output=True)

def read_script():
    with open("train_gpt.py", "r") as f:
        return f.read()

def write_script(code):
    with open("train_gpt.py", "w") as f:
        f.write(code)

def patch_replace(code, old, new, label=""):
    if old not in code:
        print(f"  WARN: patch target not found ({label})")
        return code
    return code.replace(old, new, 1)

def apply_patches(code, patch_list):
    for fn in patch_list:
        code = fn(code)
    return code

# ===== TRACK A: DEPTH-RECURRENT PATCHES =====

def patch_recurrent_basic(code):
    """Basic depth recurrence: 3 shared blocks × 3 loops = 9 effective layers.
    Reproduces PR #187 approach (expected to underperform)."""
    old = '''        self.blocks = nn.ModuleList(
            [
                Block(
                    model_dim,
                    num_heads,
                    num_kv_heads,
                    mlp_mult,
                    rope_base,
                    qk_gain_init,
                )
                for i in range(num_layers)
            ]
        )'''
    new = '''        self._num_physical = 3
        self._loops = num_layers // self._num_physical
        self.blocks = nn.ModuleList(
            [
                Block(
                    model_dim,
                    num_heads,
                    num_kv_heads,
                    mlp_mult,
                    rope_base,
                    qk_gain_init,
                )
                for i in range(self._num_physical)
            ]
        )'''
    code = patch_replace(code, old, new, "recurrent blocks")

    # Fix encoder/decoder counts
    old2 = '''        self.num_encoder_layers = num_layers // 2
        self.num_decoder_layers = num_layers - self.num_encoder_layers
        self.num_skip_weights = min(self.num_encoder_layers, self.num_decoder_layers)
        self.skip_weights = nn.Parameter(torch.ones(self.num_skip_weights, model_dim, dtype=torch.float32))'''
    new2 = '''        effective = self._num_physical * self._loops if hasattr(self, '_num_physical') else num_layers
        self.num_encoder_layers = effective // 2
        self.num_decoder_layers = effective - self.num_encoder_layers
        self.num_skip_weights = min(self.num_encoder_layers, self.num_decoder_layers)
        self.skip_weights = nn.Parameter(torch.ones(self.num_skip_weights, model_dim, dtype=torch.float32))'''
    code = patch_replace(code, old2, new2, "recurrent enc/dec")

    # Replace forward loop
    old3 = '''        # First half stores skips; second half reuses them in reverse order.
        for i in range(self.num_encoder_layers):
            x = self.blocks[i](x, x0)
            skips.append(x)
        for i in range(self.num_decoder_layers):
            if skips:
                x = x + self.skip_weights[i].to(dtype=x.dtype)[None, None, :] * skips.pop()
            x = self.blocks[self.num_encoder_layers + i](x, x0)'''
    new3 = '''        # Depth-recurrent: loop physical blocks
        layer_idx = 0
        for block_i in range(self._num_physical):
            for loop_j in range(self._loops):
                if layer_idx < self.num_encoder_layers:
                    x = self.blocks[block_i](x, x0)
                    skips.append(x)
                else:
                    dec_i = layer_idx - self.num_encoder_layers
                    if skips:
                        x = x + self.skip_weights[dec_i].to(dtype=x.dtype)[None, None, :] * skips.pop()
                    x = self.blocks[block_i](x, x0)
                layer_idx += 1'''
    code = patch_replace(code, old3, new3, "recurrent forward")
    return code

def patch_recurrent_lora(code):
    """Add LoRA rank-32 adapters per loop iteration on Q and V projections."""
    # First apply basic recurrence
    code = patch_recurrent_basic(code)

    # Add LoRA class
    old = '''class CausalSelfAttention(nn.Module):'''
    new = '''class LoRA(nn.Module):
    """Low-rank adapter: output = x + scale * B(A(x))"""
    def __init__(self, in_dim: int, out_dim: int, rank: int = 32):
        super().__init__()
        self.A = nn.Linear(in_dim, rank, bias=False)
        self.B = nn.Linear(rank, out_dim, bias=False)
        self.scale = nn.Parameter(torch.tensor(0.1, dtype=torch.float32))
        nn.init.normal_(self.A.weight, std=0.01)
        nn.init.zeros_(self.B.weight)
    def forward(self, x: Tensor) -> Tensor:
        return self.scale * self.B(self.A(x))


class CausalSelfAttention(nn.Module):'''
    code = patch_replace(code, old, new, "lora class")

    # Add per-loop LoRA adapters to GPT
    old2 = "        self._init_weights()"
    new2 = """        # Per-loop LoRA adapters for Q and V
        max_loops = self._loops if hasattr(self, '_loops') else 1
        self.q_loras = nn.ModuleList([LoRA(model_dim, model_dim) for _ in range(max_loops)])
        self.v_loras = nn.ModuleList([LoRA(model_dim, model_dim // (num_heads // num_kv_heads)) for _ in range(max_loops)])
        self._init_weights()"""
    code = patch_replace(code, old2, new2, "lora init")

    # Apply LoRA in recurrent forward — modify the loop to pass loop index
    old3 = '''                    x = self.blocks[block_i](x, x0)
                    skips.append(x)
                else:
                    dec_i = layer_idx - self.num_encoder_layers
                    if skips:
                        x = x + self.skip_weights[dec_i].to(dtype=x.dtype)[None, None, :] * skips.pop()
                    x = self.blocks[block_i](x, x0)'''
    new3 = '''                    # Apply per-loop LoRA to attention Q/V
                    block = self.blocks[block_i]
                    block.attn._q_lora = self.q_loras[loop_j]
                    block.attn._v_lora = self.v_loras[loop_j]
                    x = block(x, x0)
                    skips.append(x)
                else:
                    dec_i = layer_idx - self.num_encoder_layers
                    if skips:
                        x = x + self.skip_weights[dec_i].to(dtype=x.dtype)[None, None, :] * skips.pop()
                    block = self.blocks[block_i]
                    block.attn._q_lora = self.q_loras[loop_j]
                    block.attn._v_lora = self.v_loras[loop_j]
                    x = block(x, x0)'''
    code = patch_replace(code, old3, new3, "lora forward")

    # Apply LoRA in attention forward
    old4 = "        q = self.c_q(x).reshape(bsz, seqlen, self.num_heads, self.head_dim).transpose(1, 2)"
    new4 = """        q = self.c_q(x)
        if hasattr(self, '_q_lora') and self._q_lora is not None:
            q = q + self._q_lora(x)
        q = q.reshape(bsz, seqlen, self.num_heads, self.head_dim).transpose(1, 2)"""
    code = patch_replace(code, old4, new4, "lora q apply")

    old5 = "        v = self.c_v(x).reshape(bsz, seqlen, self.num_kv_heads, self.head_dim).transpose(1, 2)"
    new5 = """        v = self.c_v(x)
        if hasattr(self, '_v_lora') and self._v_lora is not None:
            v = v + self._v_lora(x)
        v = v.reshape(bsz, seqlen, self.num_kv_heads, self.head_dim).transpose(1, 2)"""
    code = patch_replace(code, old5, new5, "lora v apply")
    return code

def patch_recurrent_full(code):
    """Full improved recurrence: LoRA + input injection at each loop."""
    code = patch_recurrent_lora(code)

    # Add input injection: re-add x0 with learnable scale at each loop
    old = '''        layer_idx = 0
        for block_i in range(self._num_physical):
            for loop_j in range(self._loops):'''
    new = '''        self._inject_scales = getattr(self, '_inject_scales', None)
        if self._inject_scales is None:
            self._inject_scales = nn.Parameter(torch.full((self._num_physical * self._loops,), 0.1, dtype=torch.float32)).to(x.device)
        layer_idx = 0
        for block_i in range(self._num_physical):
            for loop_j in range(self._loops):
                # Input injection: re-add original embeddings
                inject_scale = self._inject_scales[layer_idx].to(dtype=x.dtype)
                x = x + inject_scale * x0'''
    code = patch_replace(code, old, new, "input injection")
    return code

PATCH_MAP_A = {
    "s31_recurrent_basic": [patch_recurrent_basic],
    "s31_recurrent_lora": [patch_recurrent_lora],
    "s31_recurrent_full": [patch_recurrent_full],
}

print(f"Track A: {len(PATCH_MAP_A)} depth-recurrent experiments defined.")

In [ ]:
import json as jsonlib, shutil, time as time_mod, subprocess, re
import glob as globmod

# ============================================================
# TRACK A: RUN DEPTH-RECURRENT EXPERIMENTS
# ============================================================
SKIP_COMPLETED = True
RESULTS_DIR = "experiments_step3_1"
os.makedirs(RESULTS_DIR, exist_ok=True)

EXPERIMENTS_A = {
    "s31_recurrent_basic": {"NUM_LAYERS": "9"},   # 3 shared × 3 loops
    "s31_recurrent_lora":  {"NUM_LAYERS": "9"},   # + LoRA rank-32
    "s31_recurrent_full":  {"NUM_LAYERS": "9", "MODEL_DIM": "640", "NUM_HEADS": "10", "NUM_KV_HEADS": "5"},
}

all_results = []
print(f"Track A: {len(EXPERIMENTS_A)} depth-recurrent experiments")
print("=" * 70)

for exp_idx, (exp_name, overrides) in enumerate(EXPERIMENTS_A.items()):
    result_path = f"{RESULTS_DIR}/{exp_name}/result.json"
    if SKIP_COMPLETED and os.path.exists(result_path):
        with open(result_path) as f:
            r = jsonlib.load(f)
        all_results.append(r)
        print(f"[A{exp_idx+1}/{len(EXPERIMENTS_A)}] SKIP {exp_name} (BPB={r.get('val_bpb', '?')})")
        continue

    config = {**BASE_CONFIG, **BATCH_SETTINGS[PROFILE], **FAST_SETTINGS}
    config.update(overrides)

    print(f"\n[A{exp_idx+1}/{len(EXPERIMENTS_A)}] === {exp_name} ===")
    patches = PATCH_MAP_A[exp_name]
    print(f"  Patches: {[fn.__name__ for fn in patches]}")

    reset_script()
    apply_base_patches()
    code = read_script()
    code = apply_patches(code, patches)
    write_script(code)

    for k, v in config.items():
        os.environ[k] = v

    env_str = " ".join(f"{k}={v}" for k, v in config.items())
    start_time = time_mod.time()
    proc = subprocess.Popen(
        f"PYTHONUNBUFFERED=1 {env_str} python train_gpt.py",
        shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in proc.stdout:
        line = line.rstrip()
        if any(k in line for k in ["step:", "val_bpb:", "peak memory", "final_int8", "warmup_step"]):
            print(f"  {line}", flush=True)
    proc.wait()
    elapsed = time_mod.time() - start_time

    if proc.returncode != 0:
        print(f"  ERROR (exit code {proc.returncode})")
        stderr = proc.stderr.read()
        if stderr:
            for l in stderr.strip().split('\n')[-10:]:
                print(f"  STDERR: {l}")
        continue

    log_files = sorted(globmod.glob("logs/*.txt"), key=os.path.getmtime)
    if not log_files:
        continue

    with open(log_files[-1]) as f:
        log_text = f.read()

    exp_result = {"experiment": exp_name, "elapsed_seconds": round(elapsed, 1), "step": 3.1, "track": "A",
                  "patches": [fn.__name__ for fn in patches], "config": config.copy()}

    m = re.search(r"final_int8_zlib_roundtrip val_loss:([\d.]+) val_bpb:([\d.]+)", log_text)
    if m:
        exp_result["val_loss"] = float(m.group(1))
        exp_result["val_bpb"] = float(m.group(2))

    m = re.search(r"Total submission size int8\+zlib: (\d+) bytes", log_text)
    if m:
        exp_result["artifact_bytes"] = int(m.group(1))

    m = re.search(r"peak memory allocated: (\d+) MiB", log_text)
    if m:
        exp_result["peak_memory_mib"] = int(m.group(1))

    steps = re.findall(r"step:(\d+)", log_text)
    if steps:
        exp_result["total_steps"] = int(steps[-1])

    exp_dir = f"{RESULTS_DIR}/{exp_name}"
    os.makedirs(exp_dir, exist_ok=True)
    shutil.copy2(log_files[-1], f"{exp_dir}/train.log")
    with open(f"{exp_dir}/result.json", "w") as f:
        jsonlib.dump(exp_result, f, indent=2)

    all_results.append(exp_result)
    print(f"  -> BPB={exp_result.get('val_bpb', '?')} | {elapsed:.0f}s")

print("\n" + "=" * 70)
print("TRACK A RESULTS")
all_results.sort(key=lambda r: r.get("val_bpb", 999))
for i, r in enumerate(all_results):
    print(f"  {i+1}. {r['experiment']:<25} BPB={r.get('val_bpb', '?'):>8} | {r.get('elapsed_seconds', 0):.0f}s")

## 6. Track B: Mamba-3 Prototype

Self-contained Mamba SSM model using the `mamba-ssm` package. Same data pipeline and evaluation as train_gpt.py but completely different architecture. No attention, linear complexity, ~2x parameter efficiency.

**Note:** If `mamba-ssm` fails to install (needs CUDA compilation), Track B will be skipped.

In [ ]:
# ============================================================
# TRACK B: MAMBA-3 PROTOTYPE
# ============================================================
# Self-contained Mamba model + training loop.
# Uses same data pipeline and BPB evaluation from train_gpt.py.

import json as jsonlib, shutil, time as time_mod, subprocess, re, sys
import glob as globmod

# Check if mamba-ssm is available
try:
    import mamba_ssm
    MAMBA_AVAILABLE = True
    print(f"mamba-ssm {mamba_ssm.__version__} available")
except ImportError:
    MAMBA_AVAILABLE = False
    print("mamba-ssm not available — Track B will be skipped")
    print("Try: pip install mamba-ssm causal-conv1d")

if MAMBA_AVAILABLE:
    # Write a self-contained Mamba training script
    mamba_script = '''
import copy, glob, io, math, os, random, sys, time, uuid, zlib
from pathlib import Path
import numpy as np
import sentencepiece as spm
import torch
import torch.nn.functional as F
from torch import Tensor, nn

# ---- Hyperparameters (from env) ----
DATA_PATH = os.environ.get("DATA_PATH", "./data/datasets/fineweb10B_sp1024")
TOKENIZER_PATH = os.environ.get("TOKENIZER_PATH", "./data/tokenizers/fineweb_1024_bpe.model")
VOCAB_SIZE = int(os.environ.get("VOCAB_SIZE", 1024))
NUM_LAYERS = int(os.environ.get("NUM_LAYERS", 8))
MODEL_DIM = int(os.environ.get("MODEL_DIM", 512))
EXPAND = int(os.environ.get("MAMBA_EXPAND", 2))
STATE_DIM = int(os.environ.get("MAMBA_STATE_DIM", 64))
TRAIN_BATCH_TOKENS = int(os.environ.get("TRAIN_BATCH_TOKENS", 262144))
TRAIN_SEQ_LEN = int(os.environ.get("TRAIN_SEQ_LEN", 1024))
VAL_BATCH_SIZE = int(os.environ.get("VAL_BATCH_SIZE", 262144))
ITERATIONS = int(os.environ.get("ITERATIONS", 2000))
WARMDOWN_ITERS = int(os.environ.get("WARMDOWN_ITERS", 400))
MAX_WALLCLOCK = float(os.environ.get("MAX_WALLCLOCK_SECONDS", 600))
LR = float(os.environ.get("LR", 0.001))
SEED = int(os.environ.get("SEED", 1337))

from mamba_ssm import Mamba2

# ---- Data loading (copied from train_gpt.py) ----
def load_data_shard(file):
    header = np.fromfile(file, dtype="<i4", count=256)
    assert header[0] == 20240520 and header[1] == 1
    num_tokens = int(header[2])
    tokens = np.fromfile(file, dtype="<u2", count=num_tokens, offset=256*4)
    return torch.from_numpy(tokens.astype(np.uint16, copy=False))

class TokenStream:
    def __init__(self, pattern):
        self.files = sorted(glob.glob(pattern))
        assert self.files, f"No files: {pattern}"
        self.file_idx = 0
        self.tokens = load_data_shard(self.files[0])
        self.pos = 0
    def take(self, n):
        chunks = []
        remaining = n
        while remaining > 0:
            avail = self.tokens.numel() - self.pos
            if avail <= 0:
                self.file_idx = (self.file_idx + 1) % len(self.files)
                self.tokens = load_data_shard(self.files[self.file_idx])
                self.pos = 0
                continue
            k = min(remaining, avail)
            chunks.append(self.tokens[self.pos:self.pos+k])
            self.pos += k
            remaining -= k
        return chunks[0] if len(chunks) == 1 else torch.cat(chunks)

# ---- BPB evaluation (copied from train_gpt.py) ----
def build_luts(sp, vocab_size, device):
    table_size = max(int(sp.vocab_size()), vocab_size)
    base_bytes = np.zeros(table_size, dtype=np.int16)
    has_space = np.zeros(table_size, dtype=np.bool_)
    is_boundary = np.ones(table_size, dtype=np.bool_)
    for tid in range(int(sp.vocab_size())):
        if sp.is_control(tid) or sp.is_unknown(tid) or sp.is_unused(tid):
            continue
        is_boundary[tid] = False
        if sp.is_byte(tid):
            base_bytes[tid] = 1
            continue
        piece = sp.id_to_piece(tid)
        if piece.startswith("\\u2581"):
            has_space[tid] = True
            piece = piece[1:]
        base_bytes[tid] = len(piece.encode("utf-8"))
    return (torch.tensor(base_bytes, dtype=torch.int16, device=device),
            torch.tensor(has_space, dtype=torch.bool, device=device),
            torch.tensor(is_boundary, dtype=torch.bool, device=device))

# ---- Mamba Model ----
class MambaLM(nn.Module):
    def __init__(self, vocab_size, num_layers, dim, expand, state_dim):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(Mamba2(d_model=dim, d_state=state_dim, d_conv=4, expand=expand))
            self.norms.append(nn.RMSNorm(dim))
        self.final_norm = nn.RMSNorm(dim)
        nn.init.normal_(self.tok_emb.weight, std=0.02)

    def forward(self, input_ids, target_ids):
        x = self.tok_emb(input_ids)
        for norm, layer in zip(self.norms, self.layers):
            x = x + layer(norm(x))
        x = self.final_norm(x).reshape(-1, x.size(-1))
        logits = F.linear(x, self.tok_emb.weight)  # tied embeddings
        return F.cross_entropy(logits.float(), target_ids.reshape(-1))

# ---- Training ----
def main():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    device = torch.device("cuda", 0)
    torch.cuda.set_device(device)

    sp = spm.SentencePieceProcessor(model_file=TOKENIZER_PATH)
    val_files = sorted(glob.glob(os.path.join(DATA_PATH, "fineweb_val_*.bin")))
    val_tokens = torch.cat([load_data_shard(f) for f in val_files])
    usable = ((val_tokens.numel() - 1) // TRAIN_SEQ_LEN) * TRAIN_SEQ_LEN
    val_tokens = val_tokens[:usable + 1]
    base_bytes_lut, has_space_lut, is_boundary_lut = build_luts(sp, VOCAB_SIZE, device)

    model = MambaLM(VOCAB_SIZE, NUM_LAYERS, MODEL_DIM, EXPAND, STATE_DIM).to(device).bfloat16()
    params = sum(p.numel() for p in model.parameters())
    print(f"Mamba model: {params:,} params, {NUM_LAYERS}L, dim={MODEL_DIM}, expand={EXPAND}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.01)
    stream = TokenStream(os.path.join(DATA_PATH, "fineweb_train_*.bin"))
    grad_accum = 8
    grad_scale = 1.0 / grad_accum

    os.makedirs("logs", exist_ok=True)
    run_id = str(uuid.uuid4())
    logfile = f"logs/{run_id}.txt"

    def log(msg):
        print(msg)
        with open(logfile, "a") as f:
            print(msg, file=f)

    log(logfile)
    log(f"model_params:{params}")

    # Eval function
    def do_eval():
        model.eval()
        total_loss, total_tokens, total_bytes = 0.0, 0, 0
        with torch.inference_mode():
            local_batch = VAL_BATCH_SIZE // grad_accum
            local_seqs = local_batch // TRAIN_SEQ_LEN
            total_seqs = (val_tokens.numel() - 1) // TRAIN_SEQ_LEN
            for start in range(0, total_seqs, local_seqs):
                end = min(start + local_seqs, total_seqs)
                raw = val_tokens[start*TRAIN_SEQ_LEN:(end*TRAIN_SEQ_LEN)+1].to(device, dtype=torch.int64)
                x, y = raw[:-1].reshape(-1, TRAIN_SEQ_LEN), raw[1:].reshape(-1, TRAIN_SEQ_LEN)
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    loss = model(x, y)
                n = y.numel()
                total_loss += loss.item() * n
                total_tokens += n
                tb = base_bytes_lut[y.reshape(-1)].to(torch.int16)
                tb += (has_space_lut[y.reshape(-1)] & ~is_boundary_lut[x.reshape(-1)]).to(torch.int16)
                total_bytes += tb.to(torch.float64).sum().item()
        model.train()
        vl = total_loss / total_tokens
        bpb = (vl / math.log(2)) * (total_tokens / total_bytes)
        return vl, bpb

    # Training loop
    t0 = time.perf_counter()
    for step in range(ITERATIONS + 1):
        elapsed_ms = 1000 * (time.perf_counter() - t0)
        if elapsed_ms > MAX_WALLCLOCK * 1000:
            log(f"step:{step}/{ITERATIONS} wallclock_cap_reached")
            break

        if step % 500 == 0 or step == ITERATIONS:
            vl, vbpb = do_eval()
            log(f"step:{step}/{ITERATIONS} val_loss:{vl:.4f} val_bpb:{vbpb:.4f} train_time:{elapsed_ms:.0f}ms")

        if step == ITERATIONS:
            break

        # LR schedule
        warmdown_start = max(ITERATIONS - WARMDOWN_ITERS, 0)
        if step >= warmdown_start:
            scale = max((ITERATIONS - step) / max(WARMDOWN_ITERS, 1), 0.0)
        else:
            scale = min(step / 20, 1.0)  # warmup
        for g in optimizer.param_groups:
            g["lr"] = LR * scale

        optimizer.zero_grad()
        train_loss = 0.0
        for _ in range(grad_accum):
            chunk = stream.take(TRAIN_BATCH_TOKENS // grad_accum + 1).to(device, dtype=torch.int64)
            x = chunk[:-1].reshape(-1, TRAIN_SEQ_LEN)
            y = chunk[1:].reshape(-1, TRAIN_SEQ_LEN)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = model(x, y)
            train_loss += loss.item()
            (loss * grad_scale).backward()
        optimizer.step()

        if step % 100 == 0:
            avg = elapsed_ms / max(step, 1)
            log(f"step:{step}/{ITERATIONS} train_loss:{train_loss/grad_accum:.4f} train_time:{elapsed_ms:.0f}ms step_avg:{avg:.2f}ms")

    log(f"peak memory allocated: {torch.cuda.max_memory_allocated()//1024//1024} MiB")

    # Save model
    torch.save(model.state_dict(), "final_model.pt")
    model_bytes = os.path.getsize("final_model.pt")
    log(f"Serialized model: {model_bytes} bytes")

    # INT8 quantize + zlib (simplified)
    state = model.state_dict()
    quant = {}
    for name, t in state.items():
        t = t.detach().cpu().float()
        if t.numel() > 65536 and t.ndim == 2:
            clip = torch.quantile(t.abs(), 0.9999, dim=1)
            scale = (clip / 127).clamp(min=1/127)
            q = (t / scale[:, None]).round().clamp(-127, 127).to(torch.int8)
            quant[name] = {"q": q, "s": scale.half()}
        else:
            quant[name] = {"v": t.half()}
    buf = io.BytesIO()
    torch.save(quant, buf)
    compressed = zlib.compress(buf.getvalue(), 9)
    with open("final_model.int8.ptz", "wb") as f:
        f.write(compressed)
    csize = len(compressed)
    log(f"Total submission size int8+zlib: {csize} bytes")

    # Final eval
    vl, vbpb = do_eval()
    log(f"final_int8_zlib_roundtrip val_loss:{vl:.4f} val_bpb:{vbpb:.4f}")

if __name__ == "__main__":
    main()
'''

    with open("train_mamba.py", "w") as f:
        f.write(mamba_script)
    print("Wrote train_mamba.py")

    # Run Mamba experiments
    MAMBA_EXPERIMENTS = {
        "s31_mamba_token": {
            "NUM_LAYERS": "8", "MODEL_DIM": "512", "MAMBA_EXPAND": "2",
            "MAMBA_STATE_DIM": "64", "LR": "0.001",
        },
        "s31_mamba_wide": {
            "NUM_LAYERS": "10", "MODEL_DIM": "640", "MAMBA_EXPAND": "2",
            "MAMBA_STATE_DIM": "64", "LR": "0.001",
        },
        "s31_mamba_deep": {
            "NUM_LAYERS": "16", "MODEL_DIM": "512", "MAMBA_EXPAND": "2",
            "MAMBA_STATE_DIM": "48", "LR": "0.0008",
        },
    }

    print(f"\nTrack B: {len(MAMBA_EXPERIMENTS)} Mamba experiments")
    print("=" * 70)

    for exp_idx, (exp_name, overrides) in enumerate(MAMBA_EXPERIMENTS.items()):
        result_path = f"{RESULTS_DIR}/{exp_name}/result.json"
        if SKIP_COMPLETED and os.path.exists(result_path):
            with open(result_path) as f:
                r = jsonlib.load(f)
            all_results.append(r)
            print(f"[B{exp_idx+1}/{len(MAMBA_EXPERIMENTS)}] SKIP {exp_name} (BPB={r.get('val_bpb', '?')})")
            continue

        config = {**BATCH_SETTINGS[PROFILE], **FAST_SETTINGS}
        config.update(overrides)

        print(f"\n[B{exp_idx+1}/{len(MAMBA_EXPERIMENTS)}] === {exp_name} ===")
        print(f"  Config: {overrides}")

        for k, v in config.items():
            os.environ[k] = v

        env_str = " ".join(f"{k}={v}" for k, v in config.items())
        start_time = time_mod.time()
        proc = subprocess.Popen(
            f"PYTHONUNBUFFERED=1 {env_str} python train_mamba.py",
            shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        for line in proc.stdout:
            line = line.rstrip()
            if any(k in line for k in ["step:", "val_bpb:", "peak memory", "final_int8", "model_params"]):
                print(f"  {line}", flush=True)
        proc.wait()
        elapsed = time_mod.time() - start_time

        if proc.returncode != 0:
            print(f"  ERROR (exit code {proc.returncode})")
            stderr = proc.stderr.read()
            if stderr:
                for l in stderr.strip().split('\n')[-10:]:
                    print(f"  STDERR: {l}")
            continue

        log_files = sorted(globmod.glob("logs/*.txt"), key=os.path.getmtime)
        if not log_files:
            continue

        with open(log_files[-1]) as f:
            log_text = f.read()

        exp_result = {"experiment": exp_name, "elapsed_seconds": round(elapsed, 1),
                      "step": 3.1, "track": "B", "config": config.copy()}

        m = re.search(r"final_int8_zlib_roundtrip val_loss:([\d.]+) val_bpb:([\d.]+)", log_text)
        if m:
            exp_result["val_loss"] = float(m.group(1))
            exp_result["val_bpb"] = float(m.group(2))

        m = re.search(r"Total submission size int8\+zlib: (\d+) bytes", log_text)
        if m:
            exp_result["artifact_bytes"] = int(m.group(1))

        m = re.search(r"peak memory allocated: (\d+) MiB", log_text)
        if m:
            exp_result["peak_memory_mib"] = int(m.group(1))

        steps = re.findall(r"step:(\d+)", log_text)
        if steps:
            exp_result["total_steps"] = int(steps[-1])

        exp_dir = f"{RESULTS_DIR}/{exp_name}"
        os.makedirs(exp_dir, exist_ok=True)
        shutil.copy2(log_files[-1], f"{exp_dir}/train.log")
        with open(f"{exp_dir}/result.json", "w") as f:
            jsonlib.dump(exp_result, f, indent=2)

        all_results.append(exp_result)
        print(f"  -> BPB={exp_result.get('val_bpb', '?')} | {elapsed:.0f}s")

# Final combined summary
print("\n" + "=" * 70)
print("STEP 3.1 ALL RESULTS (Track A + Track B)")
all_results.sort(key=lambda r: r.get("val_bpb", 999))
for i, r in enumerate(all_results):
    track = r.get("track", "?")
    print(f"  {i+1}. [{track}] {r['experiment']:<25} BPB={r.get('val_bpb', '?')} | {r.get('elapsed_seconds', 0):.0f}s")

### Compare All Experiments

Run this cell after completing multiple experiments to see a side-by-side comparison.

In [ ]:
import json as jsonlib
import matplotlib.pyplot as plt

DIRS = {
    "experiments": "Step 1",
    "experiments_step1_5": "Step 1.5",
    "experiments_step2": "Step 2",
    "experiments_step3": "Step 3",
    "/content/drive/MyDrive/parameter-golf-experiments": "Drive S1",
    "/content/drive/MyDrive/parameter-golf-experiments-step1_5": "Drive S1.5",
    "/content/drive/MyDrive/parameter-golf-experiments-step2": "Drive S2",
    "/content/drive/MyDrive/parameter-golf-experiments-step3": "Drive S3",
}

results = {}
for base_dir, label in DIRS.items():
    if not os.path.exists(base_dir):
        continue
    for fname in sorted(globmod.glob(f"{base_dir}/*/result.json")):
        with open(fname) as f:
            r = jsonlib.load(f)
        r["_source"] = label
        results[r["experiment"]] = r

results = list(results.values())

if not results:
    print("No results found.")
else:
    results.sort(key=lambda r: r.get("val_bpb", 999))
    print(f"{'#':<3} {'Experiment':<25} {'BPB':>8} {'Loss':>8} {'Source':>10}")
    print("-" * 58)
    for i, r in enumerate(results):
        print(f"{i+1:<3} {r['experiment']:<25} {r.get('val_bpb',0):>8.4f} {r.get('val_loss',0):>8.4f} {r.get('_source','?'):>10}")

    fig, ax = plt.subplots(1, 1, figsize=(14, max(6, len(results) * 0.35)))
    names = [r["experiment"] for r in results]
    bpbs = [r.get("val_bpb", 0) for r in results]
    color_map = {"Step 1": "tab:blue", "Step 1.5": "tab:cyan", "Step 2": "tab:orange",
                 "Step 3": "tab:red", "Drive S1": "tab:blue", "Drive S1.5": "tab:cyan",
                 "Drive S2": "tab:orange", "Drive S3": "tab:red"}
    colors = [color_map.get(r.get("_source", ""), "gray") for r in results]

    ax.barh(names, bpbs, color=colors)
    ax.set_xlabel("Val BPB (lower is better)")
    ax.set_title("All Steps Comparison")
    ax.invert_yaxis()
    if bpbs:
        ax.set_xlim(min(bpbs) * 0.98, max(bpbs) * 1.01)
    ax.legend(handles=[
        plt.Rectangle((0,0),1,1, fc="tab:blue", label="Step 1 (5000i)"),
        plt.Rectangle((0,0),1,1, fc="tab:cyan", label="Step 1.5 (2000i)"),
        plt.Rectangle((0,0),1,1, fc="tab:orange", label="Step 2 (2000i, EMA)"),
        plt.Rectangle((0,0),1,1, fc="tab:red", label="Step 3 (2000i, no EMA)"),
    ], loc="lower right")
    plt.tight_layout()
    plt.show()

### Save Results to Google Drive

Mount Google Drive and copy all experiment results + logs so they persist after the Colab session ends.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/parameter-golf-experiments-step3_1"
os.makedirs(DRIVE_DIR, exist_ok=True)

copied = []
if os.path.exists("experiments_step3_1"):
    for exp_name in sorted(os.listdir("experiments_step3_1")):
        src = f"experiments_step3_1/{exp_name}"
        dst = f"{DRIVE_DIR}/{exp_name}"
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            copied.append(exp_name)

print(f"Saved to: {DRIVE_DIR}")
print(f"Step 3 experiments copied: {len(copied)}")
for name in copied:
    result_file = f"{DRIVE_DIR}/{name}/result.json"
    if os.path.exists(result_file):
        with open(result_file) as f:
            r = jsonlib.load(f)
        print(f"  {name}: BPB={r.get('val_bpb', '?')}")